# Session 20a — Multi-organism essentiality data curation

**Single-purpose notebook.** Fetches binary essentiality labels and protein sequences for four bacteria (E. coli K-12, B. subtilis 168, M. pneumoniae M129, M. genitalium G37), joins with the existing Syn3A data already in the repo, and pushes two files back to the branch:

- `memory_bank/data/multiorg_essentiality/labels.csv` — `(organism, locus_tag, gene_name, essential, source)` table
- `memory_bank/data/multiorg_essentiality/sequences.fasta` — `>organism|locus_tag|gene_name` records

**No embedding, no training in this notebook.** Those are separate sessions. This is the data-curation step alone — keep it scoped so we can verify coverage + label balance before committing GPU time to embeddings.

Wall: ~5-10 minutes (mostly NCBI GenBank downloads). No GPU needed.

**Sources.** See `memory_bank/data/multiorg_essentiality/README.md` for the full citation list. In short:
- *E. coli* K-12 MG1655 — Keio collection (Baba 2006), via DEG
- *B. subtilis* 168 — Commichau 2013 / DEG
- *M. pneumoniae* M129 — Lluch-Senar 2015 (eLife)
- *M. genitalium* G37 — Glass 2006 (PNAS)
- Syn3A — Breuer 2019 (already in repo at `memory_bank/data/syn3a_essentiality_breuer2019.csv`)


In [ ]:
# Cell 1 — install minimal deps. No torch / transformers — we're not
# embedding in this notebook.
!pip install -q biopython pandas requests

In [ ]:
# Cell 2 — clone (or refresh) the project repo to branch tip.
import os, subprocess

BRANCH = "claude/syn3a-whole-cell-simulator-REjHC"
REPO_URL = "https://github.com/Nikku03/cell.git"
REPO_DIR = "/content/cell"

def _run(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.rstrip())
    if r.stderr.strip(): print(r.stderr.rstrip())
    if r.returncode != 0:
        raise RuntimeError(f"{cmd!r} exit {r.returncode}")
    return r

if not os.path.isdir(REPO_DIR):
    _run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
else:
    _run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    _run(["git", "checkout", BRANCH], cwd=REPO_DIR)
    _run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)

%cd /content/cell
import sys
sys.path.insert(0, "/content/cell")
_run(["git", "log", "--oneline", "-1"], cwd=REPO_DIR)

In [ ]:
# Cell 3 — prompt for GitHub PAT (masked).
import os, getpass
if os.environ.get("GITHUB_PAT", "").strip():
    print(f"GITHUB_PAT already set ({len(os.environ['GITHUB_PAT'])} chars)")
else:
    pat = getpass.getpass(
        "Paste your GitHub fine-grained PAT (input hidden): "
    ).strip()
    if not pat:
        raise ValueError("empty PAT — push later will fail")
    os.environ["GITHUB_PAT"] = pat
    print(f"GITHUB_PAT set ({len(pat)} chars)")

In [ ]:
# Cell 4 — per-organism configuration.
#
# Each row: (key, NCBI assembly accession, essentiality URL, fetcher).
# fetcher = a name pointing to a parse function defined in Cell 5.
#
# Multiple URLs per source are tried in order; the first one that
# responds 200 + non-empty wins. Fetch failures don't kill the
# notebook — the affected organism is just skipped and reported.
ORGANISMS = {
    "ecoli": {
        "genome_accession": "NC_000913.3",
        "taxid": 511145,
        "essentiality_urls": [
            # DEG K-12 (Keio collection) flat file mirror
            "http://origin.tubic.org/deg/public/data/DEG_data/DEG10210055.aa",
        ],
        "essentiality_parser": "deg_aa",
        "source_citation": "baba_2006_keio + deg",
    },
    "bsub": {
        "genome_accession": "NC_000964.3",
        "taxid": 224308,
        "essentiality_urls": [
            "http://origin.tubic.org/deg/public/data/DEG_data/DEG10070137.aa",
        ],
        "essentiality_parser": "deg_aa",
        "source_citation": "commichau_2013 + deg",
    },
    "mpne": {
        "genome_accession": "NC_000912.1",
        "taxid": 272634,
        "essentiality_urls": [
            # Lluch-Senar 2015 eLife — supplementary CSV mirrored via DEG
            "http://origin.tubic.org/deg/public/data/DEG_data/DEG10300184.aa",
        ],
        "essentiality_parser": "deg_aa",
        "source_citation": "lluch_senar_2015_elife + deg",
    },
    "mgen": {
        "genome_accession": "NC_000908.2",
        "taxid": 243273,
        "essentiality_urls": [
            # Glass 2006 / Hutchison 1999 — DEG entry
            "http://origin.tubic.org/deg/public/data/DEG_data/DEG1018.aa",
        ],
        "essentiality_parser": "deg_aa",
        "source_citation": "glass_2006_pnas + deg",
    },
}

for k, v in ORGANISMS.items():
    print(f"  {k:6s}  {v['genome_accession']:12s}  taxid={v['taxid']:<8d}  "
          f"src={v['source_citation']}")

In [ ]:
# Cell 5 — fetch helpers.
#
# Two fetchers, both robust to one URL failing:
#   * fetch_essentiality(urls, parser_name) -> set of essential locus_tags
#       parser_name 'deg_aa' parses DEG flat files (FASTA-like with
#       header lines that contain locus tags).
#   * fetch_genbank(accession) -> list of dicts (locus_tag, gene_name,
#       sequence) for every CDS with a translation.
#
# DEG .aa format example:
#     >DEG10210055|0001 dnaA chromosomal replication initiator b3702
#     MSLSLWQQCV...
# The b-number on the header is the K-12 locus tag we join to GenBank.
import re, time
from io import StringIO
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

from Bio import SeqIO, Entrez
Entrez.email = "cell-sim-bot@noreply.local"

_LOCUS_PATTERNS = [
    re.compile(r'\b(b\d{4})\b'),       # E. coli K-12 (b0001..)
    re.compile(r'\b(BSU\d+)\b'),       # B. subtilis 168
    re.compile(r'\b(MPN_?\d+)\b'),     # M. pneumoniae M129 (MPN001 or MPN_001)
    re.compile(r'\b(MG_?\d+)\b'),      # M. genitalium G37 (MG_001 or MG001)
]

def _http_get(url, timeout=30):
    req = Request(url, headers={"User-Agent": "cell-sim-bot/1.0"})
    with urlopen(req, timeout=timeout) as resp:
        return resp.read().decode("utf-8", errors="replace")

def _try_urls(urls):
    last_err = None
    for url in urls:
        try:
            print(f"    GET {url} ...")
            body = _http_get(url)
            if body.strip():
                return url, body
        except (HTTPError, URLError, TimeoutError) as exc:
            last_err = exc
            print(f"    fail: {type(exc).__name__}: {exc}")
    raise RuntimeError(f"all URLs failed; last err: {last_err}")

def parse_deg_aa(body):
    """DEG .aa flat file -> set of locus tags listed (every record in DEG
    is by definition essential — that's the whole DB). The locus tag is
    embedded in the header line; we try several known patterns and take
    the first hit."""
    loci = set()
    for line in body.splitlines():
        if not line.startswith(">"):
            continue
        for pat in _LOCUS_PATTERNS:
            m = pat.search(line)
            if m:
                loci.add(m.group(1))
                break
    return loci

PARSERS = {"deg_aa": parse_deg_aa}

def fetch_essentiality(urls, parser_name):
    url, body = _try_urls(urls)
    parser = PARSERS[parser_name]
    loci = parser(body)
    print(f"    parsed {len(loci)} essential locus tags")
    return loci, url

def fetch_genbank_cds(accession, max_attempts=3, retry_delay=5):
    """Use NCBI Entrez efetch to pull the GenBank record + extract every
    CDS with a translation. NCBI rate-limits at 3 req/s without API key —
    we make one call and parse locally."""
    last_err = None
    for attempt in range(max_attempts):
        try:
            handle = Entrez.efetch(
                db="nuccore", id=accession,
                rettype="gbwithparts", retmode="text",
            )
            body = handle.read()
            handle.close()
            break
        except Exception as exc:
            last_err = exc
            print(f"    NCBI attempt {attempt + 1}/{max_attempts} failed: "
                  f"{type(exc).__name__}: {exc}; sleeping {retry_delay}s")
            time.sleep(retry_delay)
    else:
        raise RuntimeError(f"NCBI efetch exhausted: {last_err}")

    rec = next(SeqIO.parse(StringIO(body), "genbank"))
    rows = []
    for f in rec.features:
        if f.type != "CDS":
            continue
        lt = (f.qualifiers.get("locus_tag") or [""])[0]
        gn = (f.qualifiers.get("gene") or [""])[0]
        tr = (f.qualifiers.get("translation") or [""])[0]
        if lt and tr:
            rows.append({
                "locus_tag": lt, "gene_name": gn,
                "sequence": tr.upper().rstrip("*"),
            })
    return rows, rec.id

In [ ]:
# Cell 6 — fetch essentiality + sequences for the 4 non-Syn3A organisms.
#
# Per organism: (1) pull DEG essential set, (2) pull GenBank CDS, (3)
# join. Genes present in the DEG set are essential=1; genes present in
# GenBank but absent from DEG are essential=0. (DEG by construction
# only lists essentials, so the negative class is the GenBank
# complement.)
#
# If essentiality fetch fails, the organism is SKIPPED — labels.csv
# will only contain orgs that fully resolved.
import pandas as pd

rows_all = []
fetch_log = {}
for orgkey, conf in ORGANISMS.items():
    print(f"\n=== {orgkey} ({conf['genome_accession']}) ===")
    log = {"organism": orgkey, "status": "fail"}
    try:
        ess_loci, ess_url = fetch_essentiality(
            conf["essentiality_urls"], conf["essentiality_parser"],
        )
        log["essentiality_url"] = ess_url
        log["n_essentials"] = len(ess_loci)
    except Exception as exc:
        log["error"] = f"essentiality: {type(exc).__name__}: {exc}"
        print(f"    SKIP {orgkey}: essentiality fetch failed")
        fetch_log[orgkey] = log
        continue

    try:
        cds_rows, gb_id = fetch_genbank_cds(conf["genome_accession"])
        log["genbank_id"] = gb_id
        log["n_cds"] = len(cds_rows)
        print(f"    {len(cds_rows)} CDS with translations")
    except Exception as exc:
        log["error"] = f"genbank: {type(exc).__name__}: {exc}"
        print(f"    SKIP {orgkey}: GenBank fetch failed")
        fetch_log[orgkey] = log
        continue

    matched = 0
    for row in cds_rows:
        # Some DEG IDs use 'b1234' but GenBank locus_tags are 'b1234'
        # too — direct match. For Mycoplasma, locus_tag formats can
        # vary (MG_001 vs MG001); check both forms.
        candidates = {row["locus_tag"]}
        if row["locus_tag"].startswith(("MG", "MPN")):
            stripped = row["locus_tag"].replace("_", "")
            candidates.add(stripped)
            if "_" not in row["locus_tag"]:
                candidates.add(
                    row["locus_tag"][:3] + "_" + row["locus_tag"][3:]
                )
        is_essential = bool(candidates & ess_loci)
        if is_essential:
            matched += 1
        rows_all.append({
            "organism": orgkey,
            "locus_tag": row["locus_tag"],
            "gene_name": row["gene_name"],
            "sequence": row["sequence"],
            "essential": int(is_essential),
            "source": conf["source_citation"],
        })
    log["matched_essentials"] = matched
    log["unmatched_in_genbank"] = len(ess_loci) - matched
    log["status"] = "ok"
    print(f"    matched: {matched} / {len(ess_loci)} DEG essentials -> "
          f"GenBank locus_tags  ({len(ess_loci) - matched} unmatched)")
    fetch_log[orgkey] = log

df_other = pd.DataFrame(rows_all)
print(f"\ntotal rows from 4 non-Syn3A organisms: {len(df_other)}")
print("\nfetch log:")
import json
print(json.dumps(fetch_log, indent=2))

In [ ]:
# Cell 7 — add Syn3A from the data already in the repo.
#
# Syn3A essentiality lives in memory_bank/data/syn3a_essentiality_breuer2019.csv;
# Syn3A sequences live in cell_sim/data/Minimal_Cell_ComplexFormation/input_data/syn3A.gb.
# The GenBank file is gitignored; we re-fetch it from upstream if missing.
from pathlib import Path
import urllib.request
from Bio import SeqIO

STAGED = Path("cell_sim/data/Minimal_Cell_ComplexFormation/input_data")
STAGED.mkdir(parents=True, exist_ok=True)
GB = STAGED / "syn3A.gb"
if not GB.exists():
    print("staging syn3A.gb from Luthey-Schulten upstream...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Luthey-Schulten-Lab/"
        "Minimal_Cell_ComplexFormation/master/input_data/syn3A.gb",
        GB,
    )

rec = next(SeqIO.parse(GB, "genbank"))
syn3a_cds = []
for f in rec.features:
    if f.type != "CDS":
        continue
    lt = (f.qualifiers.get("locus_tag") or [""])[0]
    gn = (f.qualifiers.get("gene") or [""])[0]
    tr = (f.qualifiers.get("translation") or [""])[0]
    if lt and tr:
        syn3a_cds.append({
            "locus_tag": lt, "gene_name": gn,
            "sequence": tr.upper().rstrip("*"),
        })
print(f"Syn3A CDS with translations: {len(syn3a_cds)}")

breuer = pd.read_csv("memory_bank/data/syn3a_essentiality_breuer2019.csv")
print(f"Breuer rows: {len(breuer)}  cols: {list(breuer.columns)}")
# Breuer label scheme: Essential or Quasiessential -> 1, Nonessential -> 0
label_col = (
    "essentiality" if "essentiality" in breuer.columns
    else "essentiality_class" if "essentiality_class" in breuer.columns
    else breuer.columns[1]
)
breuer["_essential"] = breuer[label_col].astype(str).str.lower().isin(
    ["essential", "quasiessential", "quasi-essential", "quasi essential"],
).astype(int)
breuer_map = dict(zip(breuer["locus_tag"], breuer["_essential"]))

syn3a_rows = []
for row in syn3a_cds:
    if row["locus_tag"] not in breuer_map:
        continue  # CDS with no Breuer label
    syn3a_rows.append({
        "organism": "syn3a",
        "locus_tag": row["locus_tag"],
        "gene_name": row["gene_name"],
        "sequence": row["sequence"],
        "essential": int(breuer_map[row["locus_tag"]]),
        "source": "breuer_2019_elife",
    })
df_syn3a = pd.DataFrame(syn3a_rows)
print(f"Syn3A rows joined to Breuer: {len(df_syn3a)}  "
      f"essential rate: {df_syn3a['essential'].mean():.3f}")

In [ ]:
# Cell 8 — combine + QC.
df_all = pd.concat([df_other, df_syn3a], ignore_index=True)
print(f"total rows: {len(df_all)}")

qc = (
    df_all.groupby("organism")
          .agg(
              n_total=("locus_tag", "count"),
              n_essential=("essential", "sum"),
              ess_rate=("essential", "mean"),
              seq_len_mean=("sequence", lambda s: int(s.str.len().mean())),
              seq_len_max=("sequence", lambda s: int(s.str.len().max())),
          )
          .reset_index()
)
print(qc.to_string(index=False))

# Sanity assertions — tunable but a sane bar.
for org in ("syn3a",):  # at minimum Syn3A must be intact
    sub = df_all[df_all["organism"] == org]
    assert len(sub) >= 400, f"{org}: only {len(sub)} rows, expected ~455"
    assert 0.0 < sub["essential"].mean() < 1.0, (
        f"{org}: degenerate essentiality column"
    )

In [ ]:
# Cell 9 — write labels.csv and sequences.fasta.
from pathlib import Path

OUT_DIR = Path("memory_bank/data/multiorg_essentiality")
OUT_DIR.mkdir(parents=True, exist_ok=True)

labels_df = df_all[
    ["organism", "locus_tag", "gene_name", "essential", "source"]
]
labels_path = OUT_DIR / "labels.csv"
labels_df.to_csv(labels_path, index=False)
print(f"wrote {labels_path}  rows={len(labels_df)}  "
      f"size={labels_path.stat().st_size/1024:.1f} KiB")

fasta_path = OUT_DIR / "sequences.fasta"
with fasta_path.open("w") as fh:
    for r in df_all.itertuples(index=False):
        gn = r.gene_name or ""
        fh.write(f">{r.organism}|{r.locus_tag}|{gn}\n")
        # 60-col wrap is the FASTA convention but a single line is
        # equally valid and faster to parse; we go single-line.
        fh.write(f"{r.sequence}\n")
print(f"wrote {fasta_path}  rows={len(df_all)}  "
      f"size={fasta_path.stat().st_size/1024:.1f} KiB")

In [ ]:
# Cell 10 — auto-commit + push.
import os, subprocess

pat = os.environ.get("GITHUB_PAT", "").strip()
if not pat:
    raise SystemExit("GITHUB_PAT not set; re-run Cell 3.")

BRANCH = "claude/syn3a-whole-cell-simulator-REjHC"
MSG = (
    "Session 20a: curate multi-organism essentiality dataset "
    "(labels + sequences for E. coli, B. subtilis, M. pneumoniae, "
    "M. genitalium, Syn3A)"
)

def run(cmd, check=True):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.rstrip())
    if r.stderr.strip(): print(r.stderr.rstrip())
    if check and r.returncode != 0:
        raise RuntimeError(f"{cmd!r} exit {r.returncode}")
    return r

run(["git", "config", "user.email", "cell-sim-bot@noreply.local"])
run(["git", "config", "user.name", "cell-sim-bot"])

to_add = [
    "memory_bank/data/multiorg_essentiality/labels.csv",
    "memory_bank/data/multiorg_essentiality/sequences.fasta",
]
run(["git", "add", "-f", *to_add])

status = run(["git", "status", "--porcelain"], check=False)
if not status.stdout.strip():
    print("nothing changed")
else:
    run(["git", "commit", "-m", MSG])
    remote = f"https://{pat}@github.com/Nikku03/cell.git"
    run(["git", "push", remote, BRANCH])
    print("\npush complete.")

run(["git", "log", "--oneline", "-3"])